In [1]:
import pandas as pd

import scripts.py.advanced
from py.advanced import solve_advanced

In this example we will use TF activity scores for selected TF.
As a target, we use PD-L1 expression.
We first load this data.

In [2]:
X_TFactivities_broad = pd.read_csv("data/X_TFactivities_Broad.csv")
y_PDL1expression_broad = pd.read_csv("data/Y_PDL1expression_Broad.csv")

For the input data X, we need rows to correspond to samples and columns to correspond to TF.
The sample ID should be the index. We do this as follows:

In [3]:
X_broad = X_TFactivities_broad.pivot(values="TF activity score", index="sample", columns=["gene"])
X_broad.head()

gene,AR,FOS,FOSL1,FOXO1,GATA2,IRF1,MYC,NFATC1,NFKB1,RELA,SOX2,SPI1,STAT1,STAT2,STAT3,STAT4,STAT6,TBX21
sample,,,,,,,,,,,,,,,,,,
SIDM00001,-3.541276,-0.330966,-0.423200,0.510541,0.107477,5.030097,-0.172900,1.333479,4.302539,3.084642,-0.911085,6.194435,2.809228,0.847982,-1.112059,2.089240,0.332944,3.859296
SIDM00005,-3.987647,-2.830855,0.704617,-1.931544,-1.384505,-1.542503,-6.123899,-1.115811,-2.922073,-3.595709,-0.818795,-2.255999,-2.875264,-1.392960,-2.582800,-0.725619,-0.008506,-0.873064
SIDM00008,-1.883773,2.343044,0.698489,0.518416,1.954996,0.978567,-6.534764,1.242436,4.513697,3.999274,2.783495,1.632235,-1.995232,1.948572,1.731846,1.712673,1.346131,1.031253
SIDM00011,-1.058458,1.624131,-0.786086,-0.078463,3.160723,-0.085951,-6.797443,4.076593,-0.185916,-0.127614,0.327426,2.338943,-0.450481,0.236227,-1.039279,0.925985,1.163530,1.095876
SIDM00015,-2.535377,-0.314842,0.275515,-0.240138,-0.871939,1.406169,-7.473478,-1.418844,0.882683,0.498863,0.269346,-1.055084,-0.067370,2.081881,-0.162005,0.091977,2.924677,-0.347223


The target y needs to be a binary series. We pick one threshold, in this case the 0.5 quantile (a.k.a. the median).

In [4]:
q = 0.5

y_series = y_PDL1expression_broad.set_index("sample")["PDL1 expression"]
y_broad = (y_series >= y_series.quantile(q))

y_broad

sample
SIDM00001     True
SIDM00005    False
SIDM00008     True
SIDM00011     True
SIDM00015     True
             ...  
SIDM01782     True
SIDM01820    False
SIDM01858    False
SIDM01859    False
SIDM01873    False
Name: PDL1 expression, Length: 849, dtype: bool

# Greedy mode

Below we illustrate the greedy method.
The following parameters need to be given:
- X (DataFrame): the (continuous) input data X
- y (Series): the binary target data
- C (int): the maximum number of iterations / rules
- D (int): the maximum number of TF per rule (in this example, D=3 is already slow)
- n_quantiles (optional int, default 9): number of thresholds to consider per TF
- beta (optional non-negative float, default 0): margin width. When set to greater than 0, the ramp loss model is used with this margin loss
- verbose (optional boolean, default True): whether to print elaborate logs during solving

There are a number of advanced features outlined in the docstring.
Note that X and y need to have a matching index (i.e. sample ID)

It returns the ruleset (list of list of Clauses) and a DataFrame with information about each round, i.e. the accuracy after iteration x.


In [5]:
from scripts.py.greedy import solve_greedy

ruleset, records = solve_greedy(X_broad, y_broad, C=3, D=2, verbose=False)

ruleset

162567it [00:07, 21809.42it/s]                                                  


['FOS>=-1.258489889098762', 'STAT2>=-0.199141937494325']


162567it [00:06, 26401.87it/s]                                                  


['FOSL1>=0.192124456300126', 'STAT2>=-0.7008762082965488']


162567it [00:05, 27986.95it/s]                                                  

['RELA>=-2.9256522978246777', '-TBX21>=0.7922907559387328']
~ Finished.
Final objective: 528.0
Final accuracy: 0.7361601884570083


[[FOS >= -1.258 Q(0.200), STAT2 >= -0.199 Q(0.500)],
 [FOSL1 >= 0.192 Q(0.799), STAT2 >= -0.701 Q(0.601)],
 [RELA >= -2.926 Q(0.101), TBX21 <= -0.792 Q(0.101)]]

In [6]:
records.head()

,it,num_total,num_pos,num_neg,rule,num_caught,num_pos_classified,frac_of_total,num_pos_left,accuracy
0,0,849,425,424,"[FOS >= -1.258 Q(0.200), STAT2 >= -0.199 Q(0.5...",0,398,0.468787,138,0.706714
1,1,451,138,313,"[FOSL1 >= 0.192 Q(0.799), STAT2 >= -0.701 Q(0....",0,37,0.043581,112,0.724382
2,2,414,112,302,"[RELA >= -2.926 Q(0.101), TBX21 <= -0.792 Q(0....",0,26,0.030624,94,0.736160


In verbose mode, useful information is printed during solving.

Usually, however, it is preferred to perform performance analysis on the result separately.
To evaluate the result on a dataset (which may also be, for instance, a test dataset), we may do the following:

In [7]:
from scripts.py.model import classify_ruleset
from sklearn.metrics import accuracy_score, precision_score, recall_score

y_hat = classify_ruleset(X_broad, ruleset) # predicted target

print("Train accuracy:", accuracy_score(y_broad, y_hat))
print("Train precision:", precision_score(y_broad, y_hat))
print("Train recall:", recall_score(y_broad, y_hat))


Train accuracy: 0.7361601884570083
Train precision: 0.7180043383947939
Train recall: 0.7788235294117647


# Advanced mode

We now illustrate how to use the advanced C++ solver.

To use the advanced solver, an executable is needed. The executable needed depends on the operating system.
* **Windows**: unpack `release-windows.zip`. Make sure all the DLL files are in the `release-windows/` folder and that `rs.exe` is in that same folder.
* **Linux**: we assume the `rs` executable is available in the `build/` folder after following the Build Instructions in `README.md`.


Below is how to use the advanced solver once the executable is in place. This example was written on a Linux machine. To use it on windows, simply add the parameter `executable='release-windows/rs.exe'`, as illustrated in the commented line.

In [12]:
import importlib
import scripts.py.advanced
importlib.reload(scripts.py.advanced)
from scripts.py.advanced import solve_advanced

ruleset, solve_info = solve_advanced(X_broad, y_broad, 3, 3, n_quantiles=9, exact_enabled=False, timelimit=20)
# For Windows users:
# ruleset, solve_info = solve_advanced(X_broad, y_broad, 3, 3, n_quantiles=9, exact_enabled=False, timelimit=20, executable='release-windows/rs.exe')


Using settings passed as parameters
Program name: build/rs
'X' file: temp/495d4633-4352-4c2c-8a9b-970acdc1ba09_X.csv
'y' file: temp/495d4633-4352-4c2c-8a9b-970acdc1ba09_y.csv
'C': 3
Output file: temp/495d4633-4352-4c2c-8a9b-970acdc1ba09_O.json
Settings file: temp/495d4633-4352-4c2c-8a9b-970acdc1ba09_S.set
'b': 0.0
Reading parameters from <temp/495d4633-4352-4c2c-8a9b-970acdc1ba09_S.set>.

Changed settings:
limits/time = 20
rs/rule_complexity_type = 0
rs/exact/enabled = FALSE
rs/heuristic/beam_widths = "1000,1000,1000"
rs/heuristic/diversity = 1
rs/exact/D = 3
rs/threshold/nonadditive = TRUE

Quantile levels: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
Clause 0: (0>=-3.48984)
Clause 1: (0>=-2.95649)
Clause 2: (0>=-2.56725)
Clause 3: (0>=-2.18629)
Clause 4: (0>=-1.83957)
Clause 5: (0>=-1.39504)
Clause 6: (0>=-0.960511)
Clause 7: (0>=-0.249344)
Clause 8: (0>=0.777586)
Clause 9: (1>=-1.85254)
Clause 10: (1>=-1.25849)
Clause 11: (1>=-0.784411)
Clause 12: (1>=-0.341346)
Clause 13: (1>=0.00

In [13]:
ruleset

[[FOS >= -0.784 Q(0.300), FOSL1 >= 0.135 Q(0.600), STAT2 >= -0.743 Q(0.300)],
 [RELA >= -2.454 Q(0.100), MYC <= -3.198 Q(0.700), TBX21 <= -0.465 Q(0.200)],
 [STAT1 >= -0.257 Q(0.700), GATA2 <= 0.923 Q(0.700), MYC <= -1.938 Q(0.800)]]

Evaluation of the ruleset works in the same way as before

In [15]:
y_hat = classify_ruleset(X_broad, ruleset) # predicted target
print(accuracy_score(y_hat, y_broad))

0.7585394581861012


If interested, additional information about the solve is available, but these are not documented here.

In [14]:
solve_info

{'C': 3,
 'binary': [['FOS>=-0.784411', 'FOSL1>=0.134828', 'STAT2>=-0.743469'],
  ['RELA>=-2.453757', '-MYC>=3.198181', '-TBX21>=0.464559'],
  ['STAT1>=-0.257192', '-GATA2>=-0.923110', '-MYC>=1.937553']],
 'dualbound': 418.0,
 'margin_width': 0.0,
 'obj': 418.0,
 'obj_cg': 396.82803045746687,
 'obj_relaxation': 1e+20,
 'pricing_steps': [{'lp_objective': 850.0,
   'pricer': 0,
   'red_cost': -360.0,
   'time': 1.557848},
  {'lp_objective': 490.0, 'pricer': 0, 'red_cost': -78.0, 'time': 2.973827},
  {'lp_objective': 466.0, 'pricer': 0, 'red_cost': -56.0, 'time': 4.383955},
  {'lp_objective': 438.0, 'pricer': 0, 'red_cost': -43.0, 'time': 5.529227},
  {'lp_objective': 415.7142857142857,
   'pricer': 0,
   'red_cost': -22.57142857142857,
   'time': 6.29703},
  {'lp_objective': 409.7987804878049,
   'pricer': 0,
   'red_cost': -23.64634146341467,
   'time': 7.031277},
  {'lp_objective': 402.93023255813955,
   'pricer': 0,
   'red_cost': -16.124031007751938,
   'time': 7.745254},
  {'lp_obje

Several other parameters are available for `solve_advanced`, described in the function's docstring.

For full control over the advanced solver, a custom settings file may be given, like those found in `data/[experiment]/`